# ResolveIQ — SLM Fine-Tuning for KB Article Generation
### IMT 526: Machine Learning Systems · Final Project · Harsh Vardhan

**What this notebook does, end to end:**
1. Downloads the Server Fault XML dump and builds train/val/test splits
2. Generates gold-standard Micro-Postmortem targets via Llama-3.3-70B (Together AI, ~$3.50)
3. Fine-tunes Llama-3.1-8B-Instruct with QLoRA SFT — 3 epochs, ~13 min on H200
4. Generates DPO rejection pairs and runs the DPO verbosity pass
5. Merges LoRA adapters into a standalone checkpoint
6. Evaluates all three model variants and produces the ablation table
7. (Optional) Runs the LIMA learning-curve hedge on 200/500/1,000 subsets

**Professor feedback incorporated:**
- 15-hour budget reference dropped; compute reported against 280-hr Tillicum allocation
- Baseline eval added as an explicit Week 8 milestone (prerequisite for the SFT gate)
- LIMA hedge (§F) trains subsets and plots a learning curve for the 1,200-example decision

**Class enhancements applied:**
- Class 11: broader LoRA target modules (q/k/v/o + gate/up/down proj)
- Class 11: LoRA sanity check at init (delta must be ~0)
- Class 13: stop-string handling to prevent over-generation
- Class 16: quantitative faithfulness score (every command traceable to input)


In [1]:
import sys, os

sys.path.insert(0, "/gpfs/projects/imt526a/hsures_cache/pip_packages")
os.environ["HF_HOME"] = "/gpfs/projects/imt526a/hsures_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOGETHER_API_KEY"] = "YOUR_TOGETHER_API_KEY"
os.environ["WANDB_API_KEY"] = "YOUR_WANDB_API_KEY"

print("✓ Environment ready")

✓ Environment ready


In [2]:
from huggingface_hub import login
login(token="HF_TOKEN_GOES_HERE")

/gpfs/projects/imt526a/conda/envs/imt526a-jupyter-torch/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

CUDA: True
GPU: NVIDIA H200
VRAM: 150.1 GB


In [ ]:
import subprocess, sys, os
result = subprocess.run(
    [sys.executable, "data/build_dataset.py",
     "--xml", "data/Posts.xml",
     "--out", "data/",
     "--n", "1500"],
    capture_output=False
)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


---
## §A — Setup and environment check

In [ ]:
# §A.4 — HuggingFace login (required to download Llama 3.1)
# You must accept the Llama 3.1 license at:
#   https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
# Then run: huggingface-cli login   (in terminal, paste your HF token)

from huggingface_hub import whoami
try:
    info = whoami()
    print(f"✓ Logged in as: {info['name']}")
except Exception:
    print("⚠️  Not logged in. Run: huggingface-cli login")


In [ ]:
# §A.5 — Project structure
from pathlib import Path

for d in ["data", "checkpoints", "results", "train", "eval"]:
    Path(d).mkdir(exist_ok=True)
print("✓ Directories ready")
print("\nProject layout:")
for f in sorted(Path(".").rglob("*.py")):
    print(f"  {f}")


---
## §B — Data: Server Fault → Incident Threads

**Source:** Stack Exchange Data Dump, Server Fault subdomain  
**License:** CC-BY-SA 4.0 — adapter weights will inherit this license on HuggingFace  
**Why Server Fault:** Public, human-authored, directly representative of IT incident resolution.  
No PII — posts use pseudonymous user IDs.


In [ ]:
# §B.1 — Download serverfault.com.7z from archive.org (~820 MB)
# Skip this cell if data/Posts.xml already exists.
import subprocess
from pathlib import Path

if not Path("data/Posts.xml").exists():
    if not Path("data/serverfault.com.7z").exists():
        print("Downloading serverfault.com.7z (~820 MB)...")
        subprocess.run([
            "wget", "-q", "--show-progress",
            "https://archive.org/download/stackexchange/serverfault.com.7z",
            "-O", "data/serverfault.com.7z"
        ], check=True)
        print("✓ Downloaded")

    print("Extracting Posts.xml...")
    subprocess.run(["7z", "e", "data/serverfault.com.7z", "Posts.xml", "-odata/", "-y"],
                   check=True, capture_output=True)
    print("✓ data/Posts.xml ready")
else:
    import os
    size_gb = os.path.getsize("data/Posts.xml") / 1e9
    print(f"✓ data/Posts.xml already exists ({size_gb:.1f} GB)")


In [ ]:
# §B.2 — Parse Posts.xml and build train/val/test splits
# Output: data/train.jsonl (1,200), data/val.jsonl (150), data/test.jsonl (150)
import subprocess, sys

result = subprocess.run(
    [sys.executable, "data/build_dataset.py",
     "--xml", "data/Posts.xml",
     "--out", "data/",
     "--n",   "1500"],
    capture_output=False
)
if result.returncode != 0:
    print("build_dataset.py failed — check the output above")
else:
    print("\n✓ Dataset built")


In [ ]:
# §B.3 — Inspect split sizes
import json
from pathlib import Path

for split in ["train", "val", "test"]:
    path = Path(f"data/{split}.jsonl")
    if path.exists():
        with open(path) as f:
            n = sum(1 for _ in f)
        print(f"  {split:6s}: {n:>5} examples")
    else:
        print(f"  {split:6s}: MISSING — run §B.2")


In [ ]:
# §B.4 — Preview one raw input thread
import json, textwrap

with open("data/train.jsonl") as f:
    ex = json.loads(f.readline())

print("=== Raw input thread (first example) ===")
print(textwrap.fill(ex["input"][:800], width=100))
print(f"\n[... total: {len(ex['input'].split())} words]")


### §B.5 — Synthetic target generation via Llama-3.3-70B-Instruct (Teacher)

**Why Llama-3.3-70B, not a closed API:**  
The Llama 3.3 Community License explicitly permits using model outputs to train derivative models.  
Closed APIs (OpenAI, Anthropic) prohibit this in their ToS.

**Cost:** ~$3.50 total (1,500 generations × ~2,500 tokens × $0.88/M).

**Quality gate:** keep only outputs where (1) all 4 headers appear in order AND (2) token count ≤ 450.  
Failures are regenerated at higher temperature.


In [ ]:
# §B.5 — Generate synthetic targets for all three splits
# Expected: ~45 min total (API call latency, not compute)
import subprocess, sys

for split in ["train", "val", "test"]:
    from pathlib import Path
    if Path(f"data/{split}_labeled.jsonl").exists():
        with open(f"data/{split}_labeled.jsonl") as f:
            n = sum(1 for _ in f)
        print(f"  ✓ {split}_labeled.jsonl already exists ({n} examples) — skipping")
        continue
    print(f"\nGenerating {split} targets...")
    subprocess.run(
        [sys.executable, "data/gen_targets.py", "--split", split, "--data", "data/"],
        check=True
    )


In [ ]:
# §B.6 — Verify filtered counts and preview one target
import json, re

HEADERS = ["## Issue Summary", "## Root Cause",
           "## Resolution Steps", "## Prevention and Action Items"]

def check_format(text):
    positions = [text.find(h) for h in HEADERS]
    return all(p != -1 for p in positions) and positions == sorted(positions)

for split in ["train", "val", "test"]:
    from pathlib import Path
    path = Path(f"data/{split}_labeled.jsonl")
    if not path.exists():
        print(f"  {split}: MISSING"); continue
    with open(path) as f:
        examples = [json.loads(l) for l in f]
    n_pass = sum(check_format(e["target"]) for e in examples)
    avg_tok = sum(len(e["target"].split()) for e in examples) / max(len(examples), 1)
    print(f"  {split:6s}: {len(examples):>5} examples | "
          f"format pass: {n_pass}/{len(examples)} | avg words/target: {avg_tok:.0f}")

print("\n=== Sample target (first training example) ===")
with open("data/train_labeled.jsonl") as f:
    ex = json.loads(f.readline())
print(ex["target"])


---
## §C — Supervised Fine-Tuning (QLoRA)

**Method:** QLoRA — model loaded in 4-bit (bitsandbytes NF4), LoRA adapters trained on top.  
**Class 11 insight:** Only ~0.3% of parameters are trained. B is initialized to zero → delta is zero at init.  
**Class 11 enhancement:** We target ALL attention + MLP projections, not just q/v_proj.  
**Expected time:** ~13–20 minutes on a single H200 (fits within a single node at 22.95 GB peak VRAM).


In [ ]:
# §C.1 — Run QLoRA SFT
# Writes checkpoints to checkpoints/sft_v1/ after each epoch.
# EarlyStoppingCallback stops automatically if val_loss plateaus.
import subprocess, sys

result = subprocess.run([sys.executable, "train/sft.py"])
if result.returncode != 0:
    print("\n⚠️  SFT failed — check output above")
else:
    print("\n✓ SFT complete")


In [ ]:
# §C.2 — Inspect the val loss curve from W&B (or local logs)
# If W&B is connected, your run is at: https://wandb.ai/YOUR_USER/
from pathlib import Path
import json

trainer_log = Path("checkpoints/sft_v1/trainer_state.json")
if trainer_log.exists():
    state = json.loads(trainer_log.read_text())
    print("Validation loss by epoch:")
    eval_entries = [e for e in state.get("log_history", []) if "eval_loss" in e]
    for e in eval_entries:
        epoch = e.get("epoch", "?")
        print(f"  Epoch {epoch:.0f}: val_loss = {e['eval_loss']:.4f}")
    if eval_entries:
        best = min(eval_entries, key=lambda e: e["eval_loss"])
        print(f"\nBest checkpoint: epoch {best.get('epoch','?'):.0f} "
              f"(val_loss = {best['eval_loss']:.4f})")
else:
    print("trainer_state.json not found — check checkpoints/sft_v1/")


In [ ]:
# §C.3 — Sanity generation: does the SFT model follow the format?
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
base = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct", revision="8c22764",
    quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16)
sft_model = PeftModel.from_pretrained(base, "checkpoints/sft_v1")
sft_tok = AutoTokenizer.from_pretrained("checkpoints/sft_v1")
sft_tok.pad_token = sft_tok.eos_token
sft_model.eval()

with open("data/val_labeled.jsonl") as f:
    sample = json.loads(f.readline())

prompt = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n"
    f"{sample['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
)
inputs = sft_tok(prompt, return_tensors="pt", truncation=True,
                 max_length=1700).to(sft_model.device)
with torch.no_grad():
    out = sft_model.generate(**inputs, max_new_tokens=512, do_sample=False,
                             pad_token_id=sft_tok.eos_token_id)
generated = sft_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

HEADERS = ["## Issue Summary","## Root Cause","## Resolution Steps","## Prevention and Action Items"]
passes = all(h in generated for h in HEADERS)
print(f"Format check: {'✓ PASS' if passes else '✗ FAIL — DPO stage may help'}")
print("\n" + generated[:1200])


---
## §D — Baseline Evaluation (Zero-Shot)

**Why this cell exists:** The professor noted that §7 Week 8 referenced a gate
("does SFT beat baseline?") but had no milestone that *produces* the baseline number.
This cell produces it. It runs the zero-shot Llama-3.1-8B on the 150-example test set
and logs results to `results/eval_baseline.json`.

Run this **before** the DPO stage so you have the comparison ready for the gate decision.


In [ ]:
# §D.1 — Zero-shot baseline on test set
# Uses the same evaluation logic as the full ablation table.
# Expected: ~15-20 min (150 inference passes)
import subprocess, sys

result = subprocess.run(
    [sys.executable, "eval/evaluate.py",
     "--models", "baseline",
     "--data",   "data/test_labeled.jsonl",
     "--out",    "results/eval_baseline.json"],
)
if result.returncode != 0:
    print("\n⚠️  Baseline eval failed")
else:
    print("\n✓ Baseline results saved to results/eval_baseline.json")


In [ ]:
# §D.2 — Print baseline results
import json

with open("results/eval_baseline.json") as f:
    res = json.load(f)

r = res.get("baseline", {})
print("=== Zero-shot baseline (Llama-3.1-8B, no fine-tuning) ===")
print(f"  Format adherence:      {r.get('format_adherence', 'N/A'):.3f}")
print(f"  BERTScore F1:          {r.get('bertscore_f1', 'N/A'):.4f}")
print(f"  Verbosity penalty rate:{r.get('verbosity_penalty_rate', 'N/A'):.3f}")
print(f"  Faithfulness score:    {r.get('faithfulness_score', 'N/A'):.3f}")
print()
print("End-of-Week-8 gate: does post-SFT BERTScore F1 exceed this number?")
print(f"  Target to beat: {r.get('bertscore_f1', 0):.4f}")


---
## §E — DPO: Direct Preference Optimization for Verbosity

**Class 12 derivation:**  
L_DPO(θ) = −E[ log σ( β·log(π_θ(y_w|x)/π_ref) − β·log(π_θ(y_l|x)/π_ref) ) ]

The Z(x) partition function cancels — no explicit reward model needed.

**Key design choice (Class 12):** Rejection pairs come from the **post-SFT model** with a
verbose system prompt, not from the base model. This ensures DPO learns conciseness,
not format compliance (which SFT already taught).

**Only run this if §D showed SFT beat the baseline.** If not, see the Risk 3 fallback in §H.


In [ ]:
# §E.1 — Generate 500 verbose rejection pairs from the SFT checkpoint
import subprocess, sys
from pathlib import Path

if Path("data/dpo_pairs.jsonl").exists():
    with open("data/dpo_pairs.jsonl") as f:
        n = sum(1 for _ in f)
    print(f"✓ dpo_pairs.jsonl already exists ({n} pairs) — skipping")
else:
    print("Generating 500 verbose rejection pairs from SFT model...")
    subprocess.run([sys.executable, "train/gen_dpo_rejects.py"], check=True)


In [ ]:
# §E.2 — Check rejection pair quality (chosen vs rejected length ratio)
import json
import statistics

with open("data/dpo_pairs.jsonl") as f:
    pairs = [json.loads(l) for l in f]

ratios = [len(p["rejected"].split()) / max(len(p["chosen"].split()), 1) for p in pairs]
print(f"Rejection pair length ratio (rejected / chosen):")
print(f"  Mean:   {statistics.mean(ratios):.2f}x")
print(f"  Median: {statistics.median(ratios):.2f}x")
print(f"  Min:    {min(ratios):.2f}x  Max: {max(ratios):.2f}x")
print()
if statistics.mean(ratios) < 1.3:
    print("⚠️  Ratio is low (<1.3x). Rejected outputs may not be verbose enough.")
    print("   Consider rerunning gen_dpo_rejects.py — DPO signal may be weak.")
else:
    print("✓ Good contrast between chosen and rejected lengths.")

# Preview one pair
print("\n=== Sample pair ===")
p = pairs[0]
print(f"CHOSEN  ({len(p['chosen'].split())} words): {p['chosen'][:300]}...")
print(f"REJECTED({len(p['rejected'].split())} words): {p['rejected'][:300]}...")


In [ ]:
# §E.3 — Run DPO training (~4 min on H200)
import subprocess, sys

result = subprocess.run([sys.executable, "train/dpo.py"])
if result.returncode != 0:
    print("\n⚠️  DPO failed — check output above. Activate §H fallback if needed.")
else:
    print("\n✓ DPO complete → checkpoints/dpo_v1/")


In [ ]:
# §E.4 — Merge LoRA adapters into a standalone checkpoint
import subprocess, sys

result = subprocess.run([sys.executable, "train/merge.py"])
if result.returncode != 0:
    print("\n⚠️  Merge failed")
else:
    from pathlib import Path
    import os
    merged = Path("checkpoints/merged_final")
    if merged.exists():
        size = sum(f.stat().st_size for f in merged.rglob("*")) / 1e9
        print(f"\n✓ Merged checkpoint: {merged}  ({size:.1f} GB)")
        print("  Zero adapter overhead at inference.")


---
## §F — Full Evaluation: Ablation Table

Three model variants × three core metrics × two class-enhanced metrics.

| Metric | Source |
|--------|--------|
| Format adherence | §5.2 — regex on 4 headers in order |
| BERTScore F1 | §5.2 — DeBERTa-xlarge-mnli encoder |
| Verbosity penalty rate | §5.2 — >15% over reference = penalized |
| Preamble clean rate | §5.6 — no chatbot filler before ## Issue Summary |
| Faithfulness score | Class 16 — every command traceable to input thread |


In [ ]:
# §F.1 — Run full evaluation (all 3 model variants)
# Expected: ~45 min total (3 models × 150 examples × inference + BERTScore)
import subprocess, sys

result = subprocess.run(
    [sys.executable, "eval/evaluate.py",
     "--models", "baseline", "post_sft", "dpo_merged",
     "--data",   "data/test_labeled.jsonl",
     "--out",    "results/eval.json"],
)
if result.returncode != 0:
    print("\n⚠️  Evaluation failed — check output above")


In [ ]:
# §F.2 — Print the ablation table
import json

with open("results/eval.json") as f:
    results = json.load(f)

cols = ["format_adherence", "bertscore_f1", "verbosity_penalty_rate",
        "preamble_clean_rate", "faithfulness_score"]
headers = ["Format", "BERTScore F1", "Verb Penalty", "No Preamble", "Faithful"]

print(f"{'Model':<22} " + " ".join(f"{h:>13}" for h in headers))
print("-" * 90)

for name in ["baseline", "post_sft", "dpo_merged"]:
    if name not in results:
        print(f"{name:<22} — not evaluated yet"); continue
    r = results[name]
    vals = [r.get(c, float('nan')) for c in cols]
    print(f"{name:<22} " + " ".join(f"{v:>13.4f}" for v in vals))


In [ ]:
# §F.3 — Visualize the improvements
import json, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

with open("results/eval.json") as f:
    results = json.load(f)

models = [m for m in ["baseline", "post_sft", "dpo_merged"] if m in results]
metrics = {
    "Format adherence":     "format_adherence",
    "BERTScore F1":         "bertscore_f1",
    "No preamble":          "preamble_clean_rate",
    "Faithfulness":         "faithfulness_score",
}
colors = {"baseline": "#94A3B8", "post_sft": "#028090", "dpo_merged": "#0A1628"}
labels = {"baseline": "Zero-shot baseline", "post_sft": "Post-SFT", "dpo_merged": "Post-SFT + DPO"}

x = np.arange(len(metrics))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 5))

for i, name in enumerate(models):
    r = results[name]
    vals = [r.get(v, 0) for v in metrics.values()]
    bars = ax.bar(x + i*width, vals, width, label=labels[name],
                  color=colors[name], alpha=0.9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics.keys())
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("ResolveIQ Ablation: Zero-shot vs SFT vs SFT+DPO")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results/ablation_chart.png", dpi=150, bbox_inches="tight")
print("✓ Chart saved: results/ablation_chart.png")
from IPython.display import Image
Image("results/ablation_chart.png")


---
## §G — Qualitative Inspection (§5.6)

Manually inspect 30 random test examples from the final (DPO-merged) model.

Two pass criteria:
1. **No preamble** — output starts directly with `## Issue Summary`
2. **No hallucinated steps** — every command/path in Resolution Steps is traceable to the input


In [ ]:
# §G.1 — Load 30 random predictions for manual inspection
import json, random, re

random.seed(42)

with open("results/dpo_merged_predictions.jsonl") as f:
    all_preds = [json.loads(l) for l in f]

sample_30 = random.sample(all_preds, min(30, len(all_preds)))

preamble_fail = [p for p in sample_30 if not p.get("preamble_clean")]
faith_fail    = [p for p in sample_30 if p.get("faithfulness", 1.0) < 0.9]

print(f"Manual inspection sample: {len(sample_30)} examples")
print(f"  Preamble failures:      {len(preamble_fail)}")
print(f"  Faithfulness < 0.9:     {len(faith_fail)}")
print(f"  Both passing:           {len(sample_30) - len(preamble_fail) - len(faith_fail)}")


In [ ]:
# §G.2 — Show preamble failures (if any)
if preamble_fail:
    print(f"=== {len(preamble_fail)} PREAMBLE FAILURES ===")
    for ex in preamble_fail[:3]:
        print("\nFirst line:", ex["prediction"].strip().split("\n")[0])
        print("Input (snippet):", ex["input"][:150])
        print("-" * 60)
else:
    print("✓ No preamble failures in sample — model starts directly with ## Issue Summary")


In [ ]:
# §G.3 — Show faithfulness failures (if any)
if faith_fail:
    print(f"=== {len(faith_fail)} FAITHFULNESS FAILURES (score < 0.9) ===")
    for ex in faith_fail[:3]:
        print(f"\nFaithfulness score: {ex.get('faithfulness', '?'):.2f}")
        # Extract resolution steps
        import re
        steps = re.search(r"## Resolution Steps\s*\n(.*?)(?=##|\Z)",
                         ex["prediction"], re.DOTALL)
        if steps:
            print("Resolution Steps:", steps.group(1)[:300])
        print("Input (snippet):", ex["input"][:200])
        print("-" * 60)
else:
    print("✓ No faithfulness failures < 0.9 in sample")


In [ ]:
# §G.4 — Show a passing example end-to-end
passing = [p for p in sample_30
           if p.get("preamble_clean") and p.get("faithfulness", 0) >= 0.9
           and p.get("format_pass")]

if passing:
    ex = passing[0]
    print("=== PASSING EXAMPLE ===")
    print("\nINPUT (first 400 chars):")
    print(ex["input"][:400])
    print("\nOUTPUT:")
    print(ex["prediction"])
    print(f"\nFormat: {'✓' if ex['format_pass'] else '✗'} | "
          f"Faithful: {ex.get('faithfulness', 0):.2f} | "
          f"Verbosity flag: {'✗ Penalized' if ex['verbosity_flag'] else '✓ OK'}")


---
## §H — LIMA Learning Curve (Professor Feedback Item #3 — Optional but Recommended)

Trains SFT on 200, 500, 1,000 subsets and plots format adherence + BERTScore F1
as a function of training-set size. Converts the Zhou et al. citation into
task-specific evidence for the 1,200-example dataset scale decision.

**Cost:** ~52 additional minutes on H200 (within the 2-hour contingency buffer).  
**When to run:** After the main SFT completes. If main SFT results are strong, this is optional.
If you need to defend the dataset size in your final report, run this.


In [ ]:
# §H.1 — Train 200/500/1000 subsets (skip if time-constrained)
import subprocess, sys

RUN_LIMA_HEDGE = True  # Set to False if you're short on time

if RUN_LIMA_HEDGE:
    result = subprocess.run(
        [sys.executable, "train/sft_subset.py", "--sizes", "200", "500", "1000"],
    )
    if result.returncode != 0:
        print("\n⚠️  Subset training failed")
    else:
        print("\n✓ Subset models trained")
else:
    print("Skipped — set RUN_LIMA_HEDGE = True to run")


In [ ]:
# §H.2 — Evaluate learning curve and plot
import subprocess, sys
from pathlib import Path

if Path("data/subset_checkpoints.json").exists():
    subprocess.run([sys.executable, "eval/learning_curve.py"])
    from IPython.display import Image
    if Path("results/learning_curve.png").exists():
        Image("results/learning_curve.png")
else:
    print("Run §H.1 first to train the subset models")


---
## §I — Risk 3 Fallback: Few-Shot Inference (if DPO stagnates)

If the DPO stage fails to reduce verbosity meaningfully compared to post-SFT,
activate this fallback: prepend 3 hand-selected concise examples to the inference
prompt. No additional training. No additional Tillicum compute.

Per the proposal: activate this at the End-of-Week-8 gate if DPO BERTScore delta
is negligible.


In [ ]:
# §I.1 — Few-shot inference using the SFT checkpoint (no DPO needed)
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Three hand-selected concise examples (from training data — pick your best ones)
FEW_SHOT_EXAMPLES = []
with open("data/train_labeled.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 3: break
        ex = json.loads(line)
        FEW_SHOT_EXAMPLES.append(ex)

def build_fewshot_prompt(thread_input, examples):
    prompt = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
    prompt += "You are ResolveIQ. Write concise Micro-Postmortems. Examples follow.\n"
    for ex in examples:
        prompt += (f"<|start_header_id|>user<|end_header_id|>\n{ex['input']}<|eot_id|>"
                   f"<|start_header_id|>assistant<|end_header_id|>\n{ex['target']}<|eot_id|>")
    prompt += (f"<|start_header_id|>user<|end_header_id|>\n"
               f"{thread_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n")
    return prompt

print(f"Few-shot prompt uses {len(FEW_SHOT_EXAMPLES)} examples")
print("This fallback requires no DPO checkpoint — uses SFT model directly.")
print("Activate by calling build_fewshot_prompt(input, FEW_SHOT_EXAMPLES) at inference time.")


---
## §J — Stretch: Expose the SLM as an MCP Tool (Class 17)

Class 17 taught that RAG and MCP are complementary layers:
- RAG = retrieval strategy (embed → similarity → top-k)
- MCP = protocol for exposing tools to a model

Here we expose the merged ResolveIQ SLM as an MCP tool that any agent (or Claude)
can call. The tool takes a raw incident thread and returns a Micro-Postmortem.

This is the production deployment story for ResolveIQ.


In [ ]:
# §J.1 — Write MCP server file (Class 17 stretch)
from pathlib import Path
Path('serve').mkdir(exist_ok=True)
mcp_lines = [
    'from fastmcp import FastMCP',
    'from vllm import LLM, SamplingParams',
    '',
    'resolveiq_mcp = FastMCP('ResolveIQ KB Generator')',
    'llm = LLM(model='checkpoints/merged_final', dtype='bfloat16', max_model_len=2048)',
    'sampling = SamplingParams(temperature=0, max_tokens=512)',
    '',
    '@resolveiq_mcp.tool()',
    'def generate_kb_article(incident_thread: str) -> str:',
    '    prompt = ("<|begin_of_text|><|start_header_id|>user<|end_header_id|>\\n"',
    '             + incident_thread + "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\\n")',
    '    return llm.generate([prompt], sampling)[0].outputs[0].text.strip()',
    '',
    'if __name__ == '__main__': import asyncio; asyncio.run(resolveiq_mcp.run())',
]
Path('serve/mcp_server.py').write_text('\n'.join(mcp_lines))
print('✓ serve/mcp_server.py written')
print('Deploy: pip install fastmcp vllm && python serve/mcp_server.py')
print('The tool generate_kb_article() is then callable by any MCP client.')

---
## §K — Results Summary and Report Checklist

Use this cell to compile everything for the final report.


In [ ]:
# §K.1 — Final results summary
import json
from pathlib import Path

print("=" * 70)
print("RESOLVEIQ — IMT 526 FINAL PROJECT RESULTS SUMMARY")
print("=" * 70)

# Ablation table
if Path("results/eval.json").exists():
    with open("results/eval.json") as f:
        results = json.load(f)
    print("\nABLATION TABLE (150-example held-out test set):")
    print(f"  {'Model':<22} {'Format':>8} {'BERTScore':>10} {'VerbPen':>9} {'Faith':>8}")
    print(f"  {'-'*62}")
    for name in ["baseline", "post_sft", "dpo_merged"]:
        if name not in results: continue
        r = results[name]
        print(f"  {name:<22} "
              f"{r.get('format_adherence',0):>8.3f} "
              f"{r.get('bertscore_f1',0):>10.4f} "
              f"{r.get('verbosity_penalty_rate',0):>9.3f} "
              f"{r.get('faithfulness_score',0):>8.3f}")
else:
    print("\n  Run §F to generate eval.json")

print()
print("DELIVERABLES CHECKLIST:")
items = {
    "data/train_labeled.jsonl":              "1,200 labeled training examples",
    "data/test_labeled.jsonl":               "150 held-out test examples",
    "data/dpo_pairs.jsonl":                  "500 DPO preference pairs",
    "checkpoints/sft_v1":                    "SFT checkpoint (LoRA adapters)",
    "checkpoints/dpo_v1":                    "DPO checkpoint",
    "checkpoints/merged_final":              "Merged final model (HuggingFace-ready)",
    "results/eval.json":                     "Ablation table (3 models × 5 metrics)",
    "results/ablation_chart.png":            "Ablation visualization",
    "results/learning_curve.png":            "LIMA hedge (optional)",
    "results/dpo_merged_predictions.jsonl":  "Per-example predictions",
    "serve/mcp_server.py":                   "MCP tool wrapper (stretch)",
}
for path, desc in items.items():
    exists = Path(path).exists()
    status = "✓" if exists else "✗"
    print(f"  {status} {desc:<45} ({path})")

print()
print("PROPOSAL CHANGES FOR FINAL REPORT:")
print("  1. §4.1: Drop 15-hr reference → report 3.3 H200-hrs vs 280-hr allocation")
print("  2. §7:   Add baseline eval milestone to Week 8 row")
print("  3. §3.4: Add LIMA hedge paragraph + reference results/learning_curve.json")
